# Computer Vision 101

เราจะสร้างระบบที่ดูภาพจากกล้อง แล้วบอกว่ามีใครกำลัง **ถือแก้ว** อยู่หรือเปล่า
โดยประกอบทีละชั้น:

1. **หาแก้วในภาพ** — ตีกรอบรอบแก้ว
2. **อ่านท่ามือ** — หาข้อนิ้ว 21 จุด แล้วดูว่ากำหรือแบ
3. **รวมกัน** — มือกำ และอยู่ตรงแก้ว = กำลังถือ

รันทีละเซลล์จากบนลงล่าง ไม่ต้องแก้โค้ด

In [ ]:
# ติดตั้งไลบรารี (ล็อกเวอร์ชันไว้ให้ผลเหมือนกันทุกเครื่อง)
!pip install -q ultralytics==8.3.* mediapipe==1.0.1

In [ ]:
# เช็กว่าเครื่องมือครบและเวอร์ชันตรง
import sys, torch, ultralytics, mediapipe, cv2
print("python     :", sys.version.split()[0])
print("torch      :", torch.__version__, "| GPU:", torch.cuda.is_available())
print("ultralytics:", ultralytics.__version__)
print("mediapipe  :", mediapipe.__version__)
assert ultralytics.__version__.startswith("8.3"), "ultralytics เวอร์ชันไม่ตรง"
print("\nพร้อมแล้ว เริ่มได้เลย (ไม่ต้องใช้ GPU)")

In [ ]:
# Colab เปิดกล้องตรง ๆ ไม่ได้ ต้องดึงภาพจากเบราว์เซอร์ผ่าน JavaScript
# run_webcam ส่งภาพทีละเฟรมให้ process_frame(bgr) -> bgr แล้ววาดผลกลับไปแสดง
import time, io
from base64 import b64decode, b64encode
import numpy as np, cv2, PIL.Image
from IPython.display import display, Javascript, clear_output
from google.colab.output import eval_js

def _webcam_js():
    display(Javascript("""
      var video, div = null, stream, captureCanvas, imgElement, labelElement;
      var pendingResolve = null, shutdown = false;

      function removeDom() {
        stream.getVideoTracks()[0].stop();
        video.remove(); div.remove();
        video = null; div = null; stream = null;
        captureCanvas = null; imgElement = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            var ctx = captureCanvas.getContext('2d');
            ctx.save(); ctx.scale(-1, 1);
            ctx.drawImage(video, -640, 0, 640, 480);
            ctx.restore();
            result = captureCanvas.toDataURL('image/jpeg', 0.8);
          }
          var lp = pendingResolve; pendingResolve = null; lp(result);
        }
      }

      async function createDom() {
        if (div !== null) return stream;
        div = document.createElement('div');
        div.style.maxWidth = '640px';
        document.body.appendChild(div);

        labelElement = document.createElement('div');
        labelElement.style.fontWeight = 'bold';
        labelElement.innerText = 'กำลังเปิดกล้อง — ถ้าเบราว์เซอร์ถามสิทธิ์ ให้กด Allow';
        div.appendChild(labelElement);

        video = document.createElement('video');
        video.width = 640;
        video.style.display = 'block';
        video.style.transform = 'scaleX(-1)';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };
        stream = await navigator.mediaDevices.getUserMedia({video: true});
        div.appendChild(video);

        imgElement = document.createElement('img');
        imgElement.width = 640;
        imgElement.style.display = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        var hint = document.createElement('div');
        hint.innerHTML = '<span style="color:red;font-weight:bold;cursor:pointer">คลิกที่ภาพเพื่อหยุดกล้อง</span>';
        div.appendChild(hint);

        video.srcObject = stream;
        await video.play();

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = 640;
        captureCanvas.height = 480;
        window.requestAnimationFrame(onAnimationFrame);
        return stream;
      }

      async function stream_frame(label, imgData) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        await createDom();
        if (label !== "") labelElement.innerText = label;
        if (imgData !== "") {
          imgElement.src = imgData;
          imgElement.style.display = 'block';
          video.style.display = 'none';
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        return {'img': result};
      }
    """))

def run_webcam(process_frame, seconds=20):
    """เปิดกล้องผ่านเบราว์เซอร์ เรียก process_frame(bgr) -> bgr ทุกเฟรม
    วาดผลกลับไปแสดง · คลิกที่ภาพ หรือครบ seconds เพื่อหยุด"""
    _webcam_js()
    overlay = ""
    deadline = time.time() + seconds
    try:
        while time.time() < deadline:
            reply = eval_js('stream_frame("", "%s")' % overlay)
            if not reply or not reply.get("img"):
                break
            jpg = b64decode(reply["img"].split(",", 1)[1])
            bgr = cv2.imdecode(np.frombuffer(jpg, np.uint8), cv2.IMREAD_COLOR)
            out = process_frame(bgr)
            _, buf = cv2.imencode(".jpg", out)
            overlay = "data:image/jpeg;base64," + b64encode(buf).decode()
    except Exception as e:
        print("เปิดกล้องไม่ได้ —", repr(e))
        print("แก้: กด Allow ตอนเบราว์เซอร์ถาม / เปลี่ยนไปใช้ Chrome / รันเซลล์นี้ใหม่")
    finally:
        try:
            eval_js("shutdown = true")
            eval_js('stream_frame("", "")')
        except Exception:
            pass

def run_video(path, process_frame, seconds=20):
    """เล่นไฟล์วิดีโอแทนกล้อง (ใช้ตอนกล้องไม่ทำงาน)"""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    for _ in range(int(fps * seconds)):
        ok, frame = cap.read()
        if not ok:
            break
        _, buf = cv2.imencode(".jpg", process_frame(frame))
        clear_output(wait=True)
        display(PIL.Image.open(io.BytesIO(buf)))
    cap.release()

### การใช้กล้อง

- ตอนเบราว์เซอร์ถามหากล้อง กด **Allow**
- **คลิกที่ภาพ** เพื่อหยุดกล้อง (หรือปล่อยให้ครบเวลาแล้วมันหยุดเอง)
- ใช้ Safari แล้วไม่ติด — เปลี่ยนไป Chrome
- ยังไม่ได้ — อัดคลิปสั้น ๆ อัปโหลดเข้ามา แล้วเปลี่ยน `run_webcam(...)` เป็น `run_video("clip.mp4", ...)`

---
## 1 · หาแก้วในภาพ

เราอยากรู้ว่า "แก้วอยู่ตรงไหน" คำตอบคือ **กรอบสี่เหลี่ยม** รอบแก้ว พร้อมค่า **ความมั่นใจ** 0–1

โมเดลที่เราจะใช้ (`yolo11n`) ผ่านการเทรนกับภาพนับแสนมาแล้ว และรู้จักของ 80 อย่าง — รวมถึง "แก้ว"
งานของเราไม่ใช่สอนมันตั้งแต่ต้น แต่ปรับให้มันแม่นกับแก้วในห้องนี้

In [ ]:
# โหลดรูปและ label
!git clone -q https://github.com/P-PrPas/tkk_workshop-data.git data
!ls data

In [ ]:
# label ต้องเป็น class 41 (cup ในชุด COCO) และพิกัดอยู่ในช่วง 0–1 — เช็กก่อนเทรน
from pathlib import Path
import matplotlib.pyplot as plt

splits = ["train", "val", "test"]
imgs = {s: sorted(Path(f"data/images/{s}").glob("*.jpg")) for s in splits}
image_names = {p.name for s in splits for p in imgs[s]}

for s in splits:
    for txt in Path(f"data/labels/{s}").glob("*.txt"):
        assert txt.with_suffix(".jpg").name in image_names, f"{txt.name} ไม่มีรูปคู่กัน"
        for line in txt.read_text().splitlines():
            if not line.strip():
                continue
            cls, *box = line.split()
            assert cls == "41", f"{txt.name}: class ต้องเป็น 41 (cup)"
            assert all(0 <= float(v) <= 1 for v in box), f"{txt.name}: พิกัดต้อง normalize 0-1"
print("label ผ่านการตรวจทั้งหมด")

# วาดกรอบจาก label ให้เห็นกับตา
all_imgs = [p for s in splits for p in imgs[s]]
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax, p in zip(axes.ravel(), all_imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    lbl = Path(str(p).replace("/images/", "/labels/")).with_suffix(".txt")
    for line in lbl.read_text().splitlines() if lbl.exists() else []:
        if not line.strip():
            continue
        _, cx, cy, bw, bh = map(float, line.split())
        x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
        x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 3)
    ax.imshow(im); ax.axis("off"); ax.set_title(p.name, fontsize=8)
for ax in axes.ravel()[len(all_imgs):]:
    ax.axis("off")
plt.tight_layout(); plt.show()

print("train:", len(imgs["train"]), "| val:", len(imgs["val"]), "| test:", len(imgs["test"]))

### สอนด้วยรูปของเราเอง

เรามีรูปแก้วในห้องนี้แค่สิบใบ ปกติน้อยเกินกว่าจะสอนของใหม่ทั้งชนิด
แต่เราไม่ได้สอนของใหม่ — โมเดลรู้จัก "แก้ว" อยู่แล้ว รูปสิบใบแค่ปรับให้มันเข้ากับแก้วและแสงตรงหน้า

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo11n.pt")          # โมเดลนี้รู้จัก cup (คลาส 41) อยู่แล้ว
model.train(data="data/cup.yaml", epochs=3, imgsz=640, batch=4, seed=0, plots=True)

### ทำไมสิบรูปถึงพอ

ถ้าเราลบความรู้เดิมของโมเดลทิ้งแล้วเริ่มสอน "แก้ว" จากศูนย์ด้วยสิบรูป — มันจะแทบไม่เจออะไรเลย

ที่มันใช้ได้ เพราะเราเก็บความรู้เดิมไว้ทั้งหมด แล้ว *ขยับ* เฉพาะส่วนที่เกี่ยวกับแก้ว
เทคนิคนี้ชื่อ **transfer learning** และงานจริงเกือบทั้งหมดทำแบบนี้ ไม่มีใครเริ่มจากศูนย์

In [ ]:
# ผลบนรูปทดสอบที่โมเดลไม่เคยเห็นตอนเทรน
import matplotlib.pyplot as plt
test_imgs = sorted(Path("data/images/test").glob("*.jpg"))
fig, axes = plt.subplots(1, len(test_imgs), figsize=(6 * len(test_imgs), 6))
for ax, p in zip(np.atleast_1d(axes), test_imgs):
    r = model(str(p), conf=0.25, classes=[41], verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); ax.axis("off"); ax.set_title(p.name)
plt.tight_layout(); plt.show()

In [ ]:
# ตัวเลขสรุปความแม่น (เฉพาะคลาส cup)
metrics = model.val(split="test", classes=[41])
print("mAP50:", round(metrics.box.map50, 3))

ตัวเลขนี้มาจากรูปแค่ไม่กี่ใบ ผิดถูกรูปเดียวก็เด้งไปมาก — อย่าไปยึดกับมัน
ให้ดูที่ภาพจริงว่ากรอบตกถูกที่หรือเปล่า นั่นบอกอะไรได้มากกว่า

### ลองกับกล้องจริง

เอาแก้วหลายแบบเข้า–ออกเฟรม เอียงดู เอามือบังบางส่วน

แล้วลองเอา **ขวดน้ำ** มาวางข้าง ๆ — มันจะไม่ขึ้นกรอบ เพราะเราสั่งให้สนใจแค่ "แก้ว" (`classes=[41]`)
ทั้งที่โมเดลก็รู้จักขวด การกำหนดขอบเขตให้แคบแบบนี้คือส่วนหนึ่งของการทำให้ระบบเชื่อถือได้

In [ ]:
def process_frame(bgr):
    r = model(bgr, conf=0.25, classes=[41], verbose=False)[0]
    return r.plot()

run_webcam(process_frame, seconds=20)
# run_video("clip.mp4", process_frame)   # กล้องไม่ทำงาน? อัปโหลดคลิปแล้วใช้บรรทัดนี้แทน

---
## 2 · อ่านท่ามือ

กรอบบอกได้แค่ว่า "มีมืออยู่ตรงนี้" แต่ไม่บอกว่ามือ *ทำท่าอะไร*

คราวนี้เราจะหา **จุดสังเกต 21 จุด** บนมือ แล้วใช้ตำแหน่งพวกนั้นตัดสินว่ากำหรือแบ

In [ ]:
# โหลดตัวตรวจจับมือ
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task
!ls -la hand_landmarker.task

In [ ]:
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

_opts = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path="hand_landmarker.task"),
    running_mode=mp_vision.RunningMode.VIDEO, num_hands=2)
landmarker = mp_vision.HandLandmarker.create_from_options(_opts)

TIPS = [4, 8, 12, 16, 20]     # ปลายนิ้วทั้งห้า
PIPS = [2, 6, 10, 14, 18]     # ข้อกลางของแต่ละนิ้ว

def count_extended(lm):
    """นับนิ้วที่เหยียด: ปลายนิ้วอยู่ไกลจากข้อมือกว่าข้อกลาง"""
    w = lm[0]
    d = lambda p: (p.x - w.x) ** 2 + (p.y - w.y) ** 2
    return sum(d(lm[t]) > d(lm[p]) for t, p in zip(TIPS, PIPS))

def hand_state(lm):
    n = count_extended(lm)
    return "FIST" if n <= 1 else "OPEN" if n >= 4 else "UNKNOWN"

### วัดระยะ ไม่ใช่วัดความสูง

วิธีแรกที่คนมักคิดคือ "ปลายนิ้วอยู่สูงกว่าข้อนิ้วไหม" — แต่พอเอียงมือหรือชี้ลง มันพังทันที

เราวัด *ระยะจากข้อมือถึงปลายนิ้ว* แทน ปลายนิ้วไกลกว่าข้อกลาง = นิ้วเหยียด วิธีนี้ทนต่อการหมุนมือ

In [ ]:
_t = [0]   # MediaPipe โหมดวิดีโอต้องการ timestamp ที่เพิ่มขึ้นเรื่อย ๆ

def hand_process_frame(bgr):
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    _t[0] += 33
    res = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), _t[0])
    h, w = bgr.shape[:2]
    for lm in (res.hand_landmarks or []):
        pts = [(int(p.x * w), int(p.y * h)) for p in lm]
        for x, y in pts:
            cv2.circle(bgr, (x, y), 4, (0, 255, 0), -1)
        cv2.putText(bgr, hand_state(lm), pts[0], cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 255, 0), 3)
    return bgr

run_webcam(hand_process_frame, seconds=20)
# run_video("clip.mp4", hand_process_frame)

กำแล้วแบช้า ๆ ดู — ตอนมืออยู่กึ่งกลาง ป้ายจะ **สั่นไปมา**
จำอาการนี้ไว้ เดี๋ยวเจอกันอีก

---
## 3 · รวมเป็น "กำลังถือแก้ว"

ไม่มีการเทรนโมเดลใหม่ เราแค่เอาผลจากสองส่วนก่อนหน้ามาต่อกันด้วยกฎเดียว:

> มือกำ **และ** กรอบมือทับกับกรอบแก้ว → กำลังถือ

In [ ]:
def boxes_overlap(a, b):
    return a[0] < b[2] and b[0] < a[2] and a[1] < b[3] and b[1] < a[3]

def hand_bbox(lm, w, h):
    xs = [p.x * w for p in lm]; ys = [p.y * h for p in lm]
    return [min(xs), min(ys), max(xs), max(ys)]

def is_holding(hbox, hand_st, cup_boxes):
    return hand_st == "FIST" and any(boxes_overlap(hbox, c) for c in cup_boxes)

In [ ]:
# รวมทั้งสองส่วนแล้วลองกับกล้อง
_tt = [0]

def combined_process_frame(bgr):
    h, w = bgr.shape[:2]
    r = model(bgr, conf=0.25, classes=[41], verbose=False)[0]
    cup_boxes = r.boxes.xyxy.tolist() if r.boxes is not None else []
    for x1, y1, x2, y2 in cup_boxes:
        cv2.rectangle(bgr, (int(x1), int(y1)), (int(x2), int(y2)), (255, 180, 0), 2)

    _tt[0] += 33
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    res = landmarker.detect_for_video(mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb), _tt[0])
    holding = False
    for lm in (res.hand_landmarks or []):
        pts = [(int(p.x * w), int(p.y * h)) for p in lm]
        for x, y in pts:
            cv2.circle(bgr, (x, y), 3, (0, 255, 0), -1)
        if is_holding(hand_bbox(lm, w, h), hand_state(lm), cup_boxes):
            holding = True
    label = "HOLDING" if holding else "NOT HOLDING"
    color = (0, 200, 0) if holding else (0, 0, 255)
    cv2.putText(bgr, label, (20, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.6, color, 4)
    return bgr

run_webcam(combined_process_frame, seconds=20)
# run_video("clip.mp4", combined_process_frame)

---
## ของจริงยากกว่านี้

สิ่งที่เราเพิ่งสร้างพอเห็นผลได้ แต่ยังห่างจากของที่ใช้งานจริงมาก ลองสังเกตเอง:

- ป้ายสั่นตลอด — ระบบตัดสินใหม่ทุกเฟรม ไม่จำเฟรมก่อนหน้า
- แก้วหายไปแวบเดียวแล้วกลับมา ระบบนับเป็นใบใหม่
- ช้า
- ถือแบบประคอง (ไม่กำ) ระบบไม่นับ เพราะกฎบังคับว่าต้องกำ
- กล้องหลุดแล้วทุกอย่างค้าง

**การทำให้พอเห็นผลนั้นเร็ว การทำให้เชื่อถือได้ต่างหากที่ยาก**
ห้าข้อนี้คือช่องว่างนั้น — เดี๋ยวเราจะดูตัวที่แก้ครบทุกข้อ